In [2]:
#from google.colab import drive
#drive.flush_and_unmount()

import os
#from google.colab import drive
#drive.mount('/content/drive')

drive_root = './implementation'
os.chdir(drive_root)
files = os.listdir(drive_root)
print(files)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
['algorithm', 'data_preprocessing', 'experiment', 'eval_metrics', 'loss_function']


# COIL20

In [15]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import data_preprocessing.coil20 as coil20
from importlib import reload
reload(coil20)

# training process
import algorithm.algo1 as algo1
reload(algo1)

# evaluation
import algorithm.kmeans as kmeans
import eval_metrics.ari

# loss function
import loss_function.origin as origin
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

import sys
if 'ipykernel' in sys.argv[0]:
    sys.argv = [arg for arg in sys.argv if arg != '-f']

import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args, unknown = parser.parse_known_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

[Using device] cuda


In [10]:
import random

# 设定随机种子
random.seed(42)
torch.manual_seed(42)

## CLUESTER_NUM = 2

In [6]:
CLUSTER_NUM = 2
NUM_EXPERIMENTS = 10  #进行10次实验，选择不同的实验类别
LAMBDA_LIST = [0.5, 1, 1.5, 2, 2.5, 3]  #采用不同的lambda进行试验

In [ ]:
import numpy as np

avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    # 读取数据
    mat, label = coil20.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = eval_metrics.ari.ari(label, pred_labels)
        nmi = eval_metrics.nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi)
        else:
            avg_nmi[LAMBDA] = [nmi]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

******************************
[Using Classes] [11 17]
******************************
[Using LAMBDA] 0.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 1
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 1.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 2
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 2.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 3
[Accuracy] 1.0
[Mutual Information] 1.0
******************************
[Using Classes] [ 5 20]
******************************
[Using LAMBDA] 0.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 1
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 1.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 2
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 2.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 3
[Accuracy] 1.0
[Mutual Information] 1.0
******************************
[Using Classes] [7 9]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.5388593719661801

## CLUSTER_NUM = 5

In [ ]:
CLUSTER_NUM = 5

In [ ]:
import numpy as np

avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    # 读取数据
    mat, label = coil20.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = eval_metrics.ari.ari(label, pred_labels)
        nmi = eval_metrics.nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi)
        else:
            avg_nmi[LAMBDA] = [nmi]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

******************************
[Using Classes] [ 2  3  4 12 13]
******************************
[Using LAMBDA] 0.5


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.6690038116761915
[Mutual Information] 0.7441742344046177
[Using LAMBDA] 1
[Accuracy] 0.6690038116761915
[Mutual Information] 0.7441742344046177
[Using LAMBDA] 1.5
[Accuracy] 0.6690038116761915
[Mutual Information] 0.7441742344046177
[Using LAMBDA] 2
[Accuracy] 0.6660710498497877
[Mutual Information] 0.7425592749067994
[Using LAMBDA] 2.5
[Accuracy] 0.6690038116761915
[Mutual Information] 0.7441742344046177
[Using LAMBDA] 3
[Accuracy] 0.6690038116761915
[Mutual Information] 0.7441742344046177
******************************
[Using Classes] [ 4  7 12 13 19]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.6530263702660323
[Mutual Information] 0.7427964395578838
[Using LAMBDA] 1


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.6547144553844794
[Mutual Information] 0.7437157963182932
[Using LAMBDA] 1.5
[Accuracy] 0.6530263702660323
[Mutual Information] 0.7427964395578838
[Using LAMBDA] 2
[Accuracy] 0.6530263702660323
[Mutual Information] 0.7427964395578838
[Using LAMBDA] 2.5


...

K-means algorithm did not converge in 200 iterations
[Accuracy] 0.8904390286436762
[Mutual Information] 0.9259046521006453
******************************
[Using Classes] [ 1  4 14 15 18]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.8062601187263896
[Mutual Information] 0.8921322072364292
[Using LAMBDA] 1
[Accuracy] 0.8062601187263896
[Mutual Information] 0.8921322072364292
[Using LAMBDA] 1.5


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.8062601187263896
[Mutual Information] 0.8921322072364292
[Using LAMBDA] 2


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.8062601187263896
[Mutual Information] 0.8921322072364292
[Using LAMBDA] 2.5
[Accuracy] 0.8062601187263896
[Mutual Information] 0.8921322072364292
[Using LAMBDA] 3
[Accuracy] 0.8062601187263896
[Mutual Information] 0.8921322072364292
Average Accuracy for LAMBDA=0.5: 0.6747986373232998
Average Mutual Information for LAMBDA=0.5: 0.7642698999113972
Average Accuracy for LAMBDA=1: 0.6766182932991487
Average Mutual Information for LAMBDA=1: 0.7666368771679438
Average Accuracy for LAMBDA=1.5: 0.6266402774062161
Average Mutual Information for LAMBDA=1.5: 0.7149484257711836
Average Accuracy for LAMBDA=2: 0.6261122842361773
Average Mutual Information for LAMBDA=2: 0.7137467334375139
Average Accuracy for LAMBDA=2.5: 0.612381171676751
Average Mutual Information for LAMBDA=2.5: 0.6988543459099776
Average Accuracy for LAMBDA=3: 0.5271931087895958
Average Mutual Information for LAMBDA=3: 0.6077493923642353


## CLUSTER_NUM = 10

In [18]:
CLUSTER_NUM = 10

In [19]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = coil20.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

******************************
[Using Classes] [ 1  4  7 10 11 12 14 15 16 17]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.0857536261711327
[Mutual Information] 0.2105926528841831
[Using LAMBDA] 1
[Accuracy] 0.09882929208519663
[Mutual Information] 0.23320125866552188
[Using LAMBDA] 1.5
[Accuracy] 0.13432709614154895
[Mutual Information] 0.2972164402282466
[Using LAMBDA] 2
[Accuracy] 0.16594442372504764
[Mutual Information] 0.3545066849564567
[Using LAMBDA] 2.5
[Accuracy] 0.17211798385638352
[Mutual Information] 0.3551284243292532
[Using LAMBDA] 3
[Accuracy] 0.25518721445937587
[Mutual Information] 0.4721340873403045
******************************
[Using Classes] [ 1  5  6  7  9 15 16 17 18 19]
******************************
[Using LAMBDA] 0.5


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.5318050004508726
[Mutual Information] 0.6679201429282913
[Using LAMBDA] 1


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.5490267496108299
[Mutual Information] 0.6897985636173114
[Using LAMBDA] 1.5
[Accuracy] 0.08138874425505979
[Mutual Information] 0.2158547274007448
[Using LAMBDA] 2
[Accuracy] 0.11802109796674753
[Mutual Information] 0.2647984471835244
[Using LAMBDA] 2.5
[Accuracy] 0.13474740848425398
[Mutual Information] 0.3033904727214847
[Using LAMBDA] 3
[Accuracy] 0.5567925482374926
[Mutual Information] 0.6875764575100519
******************************
[Using Classes] [ 2  3  4  7  8 11 14 16 18 19]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.5265417232016828
[Mutual Information] 0.7005259003236641
[Using LAMBDA] 1
[Accuracy] 0.10748008467832197
[Mutual Information] 0.26620974085246385
[Using LAMBDA] 1.5
[Accuracy] 0.14775618237298263
[Mutual Information] 0.32776821348509483
[Using LAMBDA] 2
[Accuracy] 0.5273448525498557
[Mutual Information] 0.6990773400411556
[Using LAMBDA] 2.5
[Accuracy] 0.13386177942812708
[Mutua

...

K-means algorithm did not converge in 200 iterations
[Accuracy] 0.6875859218134447
[Mutual Information] 0.8437344027488203
[Using LAMBDA] 1
[Accuracy] 0.14311684032561156
[Mutual Information] 0.29796914263772734
[Using LAMBDA] 1.5
[Accuracy] 0.1360750172829689
[Mutual Information] 0.3041473485351006
[Using LAMBDA] 2


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.6875859218134447
[Mutual Information] 0.8437344027488204
[Using LAMBDA] 2.5
[Accuracy] 0.15375211727357685
[Mutual Information] 0.33410855850536253
[Using LAMBDA] 3
[Accuracy] 0.16066459450748
[Mutual Information] 0.35311359520094615
******************************
[Using Classes] [ 3  5  7  8 10 13 14 17 18 20]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.7866720789026946
[Mutual Information] 0.8582910724673236
[Using LAMBDA] 1
[Accuracy] 0.1534287157523612
[Mutual Information] 0.308200251113361
[Using LAMBDA] 1.5
[Accuracy] 0.12688221637115174
[Mutual Information] 0.29357260671633945
[Using LAMBDA] 2
[Accuracy] 0.14021922677822432
[Mutual Information] 0.2959160252210096
[Using LAMBDA] 2.5
[Accuracy] 0.1668407211149621
[Mutual Information] 0.3526986095360213
[Using LAMBDA] 3


/content/drive/MyDrive/5002_project/implementation/algorithm/kmeans.py:41: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 200 iterations
[Accuracy] 0.6876438695358251
[Mutual Information] 0.8309419259497446
******************************
[Using Classes] [ 2  9 10 12 13 14 15 16 18 19]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.6992244183738023
[Mutual Information] 0.8165350160183289
[Using LAMBDA] 1
[Accuracy] 0.08909184360563026
[Mutual Information] 0.2241818763680425
[Using LAMBDA] 1.5
[Accuracy] 0.1635816503177445
[Mutual Information] 0.3351762294699001
[Using LAMBDA] 2
[Accuracy] 0.0900107628243714
[Mutual Information] 0.22724503759127912
[Using LAMBDA] 2.5
[Accuracy] 0.2972709769286882
[Mutual Information] 0.4687854410158578
[Using LAMBDA] 3
[Accuracy] 0.09731086555919108
[Mutual Information] 0.23900098506296022
Average Accuracy for LAMBDA=0.5: 0.5297449955978636
Average Mutual Information for LAMBDA=0.5: 0.6584900578157723
Average Accuracy for LAMBDA=1: 0.26617001213754576
Average Mutual Information for LAMBDA=1: 0.4072074183341326
Average Ac

## CLUSTER_NUM = 20

In [1]:
CLUSTER_NUM = 20

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = coil20.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

# REUTERS

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import data_preprocessing.reuters as reuters
from importlib import reload
reload(reuters)

# training process
import algorithm.algo1 as algo1
reload(algo1)

# evaluation
import algorithm.kmeans as kmeans
import eval_metrics.ari

# loss function
import loss_function.origin as origin
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

import sys
if 'ipykernel' in sys.argv[0]:
    sys.argv = [arg for arg in sys.argv if arg != '-f']

import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args, unknown = parser.parse_known_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda


[Using device] cuda


In [4]:
import random

# 设定随机种子
random.seed(42)
torch.manual_seed(42)

## CLUSTER_NUM = 2

In [7]:
CLUSTER_NUM = 2
NUM_EXPERIMENTS = 10  #进行10次实验，选择不同的实验类别
LAMBDA_LIST = [0.5, 1, 1.5, 2, 2.5, 3]  #采用不同的lambda进行试验

In [10]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = reuters.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

Reading reuters21578 dataset...
Counting the number of occurrences of each topic...
Checking documents with topics that occur less than 10 times...
Removing documents with topics that occur less than 10 times...
******************************
[Using Classes] [3 9]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.9509818764037117
[Mutual Information] 0.9045898373882779
[Using LAMBDA] 1
[Accuracy] 0.9509818764037117
[Mutual Information] 0.9045898373882779
[Using LAMBDA] 1.5
[Accuracy] 0.9509818764037117
[Mutual Information] 0.9045898373882779
[Using LAMBDA] 2
[Accuracy] 0.9509818764037117
[Mutual Information] 0.9045898373882779
[Using LAMBDA] 2.5
[Accuracy] 0.9509818764037117
[Mutual Information] 0.9045898373882779
[Using LAMBDA] 3
[Accuracy] 0.9509818764037117
[Mutual Information] 0.9045898373882779
Reading reuters21578 dataset...
Counting the number of occurrences of each topic...
Checking documents with topics that occur less than 10 times...
Removing documents with topi

## CLUSTER_NUM = 5

In [11]:
CLUSTER_NUM = 5

In [12]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = reuters.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

Reading reuters21578 dataset...
Counting the number of occurrences of each topic...
Checking documents with topics that occur less than 10 times...
Removing documents with topics that occur less than 10 times...
******************************
[Using Classes] [ 9 10 27 32 41]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.6716398070294066
[Mutual Information] 0.7682025404279794
[Using LAMBDA] 1
[Accuracy] 0.6716398070294066
[Mutual Information] 0.7682025404279795
[Using LAMBDA] 1.5
[Accuracy] 0.026710382254453973
[Mutual Information] 0.05440519423976856
[Using LAMBDA] 2
[Accuracy] 0.1703636724605216
[Mutual Information] 0.19463964978879109
[Using LAMBDA] 2.5
[Accuracy] 0.2005431622757978
[Mutual Information] 0.3073071738742769
[Using LAMBDA] 3
[Accuracy] 0.14855959974903563
[Mutual Information] 0.1972783831744551
Reading reuters21578 dataset...
Counting the number of occurrences of each topic...
Checking documents with topics that occur less than 10 times...
Removing doc

## CLUSTER_NUM = 10

In [13]:
CLUSTER_NUM = 10

In [14]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = reuters.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

Reading reuters21578 dataset...
Counting the number of occurrences of each topic...
Checking documents with topics that occur less than 10 times...
Removing documents with topics that occur less than 10 times...
******************************
[Using Classes] [ 3 11 12 15 16 20 23 24 26 35]
******************************
[Using LAMBDA] 0.5


KeyboardInterrupt: 

## CLUSTER_NUM = 20

In [38]:
CLUSTER_NUM = 20

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = reuters.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

In [ ]:
prompt = f'''I will give you an INTENTION.
            You need to give 2 or 3 words as ABSTRACT INTENTION.
            You must return your answer in the following format: Word1,Word2,Word3,....
            These abstract intention words should fulfill the following requirements.
            1. The ABSTRACT INTENTION WORDS can well represent represent the INTENTION.
            2. The ABSTRACT INTENTION WORDS should have minimal ambiguity and avoid multiple less relevant meanings. For example, "spring" is not a good abstract intention word because it can represent both a coiled metal device and the season of the year.
            3. The ABSTRACT INTENTION WORDS should be relatively concrete.
            4. The ABSTRACT INTENTION WORDS for the same INTENTION should not be semantically similar with each other.
            You can refer to the following example.
            Example1:
            INTENTION: PersonX enjoys exercising in the gym.
            Your answer: fitness training,health improvement,gym workouts
            Example2:
            INTENTION: PersonX enjoys hiking.
            Your answer: outdoor adventure,nature exploration, sports
            Here is the INTENTION:
            INTENTION: {intention}.
            Your answer:
            '''

In [ ]:
import json
from time import sleep
from tqdm import tqdm
import random
import argparse
import logging
from collections import Counter
import matplotlib.pyplot as plt
from gpt4_prompting import get_message_from_api as gpt4_api
from gpt35_prompting import get_message_from_api as gpt35_api


def generate(args):
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    file_handler = logging.FileHandler('generate_log.log')
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logging.getLogger().addHandler(file_handler)

    with open(args.input_file, "r") as file:
        lines = file.readlines()
    sessions = []
    for line in lines:
        dict_obj = json.loads(line)
        sessions.append(dict_obj)
    if args.sample_num:
        sessions = random.sample(sessions, args.sample_num)

    with open(args.output_file, "w") as file:
        for session in tqdm(sessions):
            intention = session['Intentions']
            tmp = []
            for intention in intention:
                prompt = f'''I will give you an INTENTION.
                You need to give 2 or 3 words as ABSTRACT INTENTION.
                You must return your answer in the following format: Word1,Word2,Word3,....
                These abstract intention words should fulfill the following requirements:
                1. The ABSTRACT INTENTION WORDS should accurately represent the INTENTION.
                2. The ABSTRACT INTENTION WORDS should have minimal ambiguity and avoid multiple less relevant meanings. For example, "spring" is not a good abstract intention word because it can represent both a coiled metal device and the season of the year.
                3. The ABSTRACT INTENTION WORDS should have a good level of abstraction, balancing specificity and generality.
                4. The ABSTRACT INTENTION WORDS for the same INTENTION should not be semantically similar to each other. For example, health and wellness are two close synonyms, so they can't be used together.
                You can refer to the following examples:
                Example 1:
                INTENTION: PersonX enjoys exercising in the gym.
                Your answer: fitness training, health improvement, gym workouts
                Example 2:
                INTENTION: PersonX enjoys hiking.
                Your answer: outdoor adventure, nature exploration, sports
                Here is the INTENTION:
                INTENTION: {intention}.
                Your answer:
                '''
                try:
                    if args.api_type == "gpt-35":
                        a, prompt_tokens, completion_tokens = gpt35_api(prompt)
                    elif args.api_type == "gpt-4":
                        a, prompt_tokens, completion_tokens = gpt4_api(prompt)
                    else:
                        raise ValueError("Invalid API type.")
                except Exception as e:
                    logging.error(f"Error occurred on element intention: {intention}. Error message: {e}")
                    sleep(2)
                    continue

                answers = a.split(",")
                for item in answers:
                    item = item.strip()
                tmp.append({
                    "INTENTION": intention,
                    "ABSTRACT INTENTION": answers,
                    "prompt_tokens": prompt_tokens,
                    "completion_tokens": completion_tokens,
                    "api_type": args.api_type
                })
            session['abstract_generation_result'] = tmp
            json.dump(session, file)
            file.write("\n")
    file.close()
    return


def static(results):
    pricing_35 = {
        "prompt_tokens": 0.0015,
        "completion_tokens": 0.002,
    }
    pricing_4 = {
        "prompt_tokens": 0.03,
        "completion_tokens": 0.06,
    }
    total_cost = 0
    abstract_intentions = []
    # 统计conceptualization之间的交叉情况
    for item in results:
        abstract_intentions.extend(item['ABSTRACT INTENTION'])
        if item['api_type'] == "gpt-4":
            total_cost += item['prompt_tokens'] * pricing_4['prompt_tokens']
            total_cost += item['completion_tokens'] * pricing_4['completion_tokens']
        elif item['api_type'] == "gpt-35":
            total_cost += item['prompt_tokens'] * pricing_35['prompt_tokens']
            total_cost += item['completion_tokens'] * pricing_35['completion_tokens']
    counter = Counter(abstract_intentions)
    count = 0
    for key, value in counter.items():
        if value >= 2:
            count += 1
    ratio = count / len(counter.keys())
    # 出现两次及以上的abstract intention word占所有word的比例为： 0.29295426452410384
    # conceptualization之间的交叉比例： 0.018081677070159796
    print('出现两次及以上的abstract intention word占所有word的比例为：', ratio)
    count = 0
    for i in range(len(results)):
        for j in range(i+1, len(results)):
            if len(set(results[i]['ABSTRACT INTENTION']) & set(results[j]['ABSTRACT INTENTION'])) > 0:
                count += 1
    print('conceptualization之间的交叉比例：', count/(len(results)*(len(results)-1)/2))


def main():
    random.seed(8)
    parser = argparse.ArgumentParser()
    parser.add_argument('--input_file', type=str, default="./week4/gpt-35-turbo_answer_0_intentions.json", help="Path to the input file.")
    parser.add_argument('--output_file', type=str, default="./week4/abstract_intention_test_zrb.json", help="Path to the output file.")
    parser.add_argument('--sample_num', type=int, default=None, help="Sample number of total pairs.")
    parser.add_argument('--api_type', type=str, default="gpt-35", help="API type to use.")
    args = parser.parse_args()
    generate(args)


if __name__=="__main__":
    main()

# TDT2

In [47]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import data_preprocessing.tdt2 as tdt2
from importlib import reload
reload(tdt2)

# training process
import algorithm.algo1 as algo1
reload(algo1)

# evaluation
import algorithm.kmeans as kmeans
import eval_metrics.ari

# loss function
import loss_function.origin as origin
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

import sys
if 'ipykernel' in sys.argv[0]:
    sys.argv = [arg for arg in sys.argv if arg != '-f']

import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args, unknown = parser.parse_known_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

[Using device] cuda


## CLUSTER_NUM = 2

In [48]:
CLUSTER_NUM = 2
NUM_EXPERIMENTS = 10  #进行10次实验，选择不同的实验类别
LAMBDA_LIST = [0.5, 1, 1.5, 2, 2.5, 3]  #采用不同的lambda进行试验

In [49]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = tdt2.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

******************************
[Using Classes] [ 7 16]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.35251166200843675
[Mutual Information] 0.4011088760426449
[Using LAMBDA] 1
[Accuracy] 0.44196034231179404
[Mutual Information] 0.47340400626275564
[Using LAMBDA] 1.5
[Accuracy] 0.9896649365436763
[Mutual Information] 0.9746185704397003
[Using LAMBDA] 2
[Accuracy] 0.9896649365436763
[Mutual Information] 0.9746185704397003
[Using LAMBDA] 2.5
[Accuracy] 0.9896649365436763
[Mutual Information] 0.9746185704397003
[Using LAMBDA] 3
[Accuracy] 0.9896649365436763
[Mutual Information] 0.9746185704397003
******************************
[Using Classes] [ 5 12]
******************************
[Using LAMBDA] 0.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 1
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 1.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 2
[Accuracy] 1.0
[Mutual Information] 1.0
[Using LAMBDA] 2.5
[Accuracy] 1.0
[Mutual Information] 1.0
[Using

## CLUSTER_NUM = 5

In [50]:
CLUSTER_NUM = 5

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = tdt2.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

******************************
[Using Classes] [ 6 13 15 16 20]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.838848959066912
[Mutual Information] 0.8579230097953705
[Using LAMBDA] 1
[Accuracy] 0.9969227404914707
[Mutual Information] 0.9910293365521622
[Using LAMBDA] 1.5
[Accuracy] 0.9969227404914707
[Mutual Information] 0.9910293365521622
[Using LAMBDA] 2
[Accuracy] 0.9969227404914707
[Mutual Information] 0.9910293365521622
[Using LAMBDA] 2.5
[Accuracy] 0.9969227404914707
[Mutual Information] 0.9910293365521622
[Using LAMBDA] 3
[Accuracy] 0.9969227404914707
[Mutual Information] 0.9910293365521622
******************************
[Using Classes] [ 4  9 13 16 28]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.6153828068781471
[Mutual Information] 0.7942385412754249
[Using LAMBDA] 1
[Accuracy] 0.6121149455230327
[Mutual Information] 0.7935329296238155
[Using LAMBDA] 1.5
[Accuracy] 0.6130761853870529
[Mutual Information] 0.7946558253081294
[Using LAMBDA] 2
[

## CLUSTER_NUM = 10

In [ ]:
CLUSTER_NUM = 10

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = tdt2.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

## CLUSTER_NUM = 20

In [18]:
CLUSTER_NUM = 20

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = tdt2.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

# CIFAR10

In [16]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import data_preprocessing.cifar10 as cifar10
from importlib import reload
reload(cifar10)

# training process
import algorithm.algo1 as algo1
reload(algo1)

# evaluation
import algorithm.kmeans as kmeans
import eval_metrics.ari

# loss function
import loss_function.origin as origin
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

import sys
if 'ipykernel' in sys.argv[0]:
    sys.argv = [arg for arg in sys.argv if arg != '-f']

import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args, unknown = parser.parse_known_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

[Using device] cuda


## CLUSTER_NUM = 2

In [20]:
CLUSTER_NUM = 2
NUM_EXPERIMENTS = 10  #进行10次实验，选择不同的实验类别
LAMBDA_LIST = [0.5, 1, 1.5, 2, 2.5, 3]  #采用不同的lambda进行试验

In [21]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = cifar10.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

******************************
[Using Classes] [4 7]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.02507494746185293
[Mutual Information] 0.01974462798722272
[Using LAMBDA] 1
[Accuracy] 0.019891231844878104
[Mutual Information] 0.015790386385616143
[Using LAMBDA] 1.5
[Accuracy] 0.015269358391633826
[Mutual Information] 0.012362795529078263
[Using LAMBDA] 2
[Accuracy] 0.02506335430722241
[Mutual Information] 0.019549140103013278
[Using LAMBDA] 2.5
[Accuracy] 0.02506335430722241
[Mutual Information] 0.019549140103013278
[Using LAMBDA] 3
[Accuracy] 0.02506335430722241
[Mutual Information] 0.019549140103013278
******************************
[Using Classes] [1 6]
******************************
[Using LAMBDA] 0.5
[Accuracy] 0.004742086037739439
[Mutual Information] 0.004635344454457663
[Using LAMBDA] 1
[Accuracy] 0.00903151276668283
[Mutual Information] 0.007768234831729513
[Using LAMBDA] 1.5
[Accuracy] 0.00903151276668283
[Mutual Information] 0.007768234831729513
[Using LAM

## CLUSTER_NUM = 5

In [ ]:
CLUSTER_NUM = 5

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = cifar10.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

## CLUSTER_NUM = 10

In [1]:
CLUSTER_NUM = 10

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = cifar10.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')

## CLUSTER_NUM = 20

In [ ]:
CLUSTER_NUM = 20

In [ ]:
import numpy as np
avg_acc = {}
avg_nmi = {}

for _ in range(NUM_EXPERIMENTS):
    from eval_metrics import nmi
    from eval_metrics import ari
    # 读取数据
    mat, label = cifar10.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, random_choose_class=True)
    mat = torch.tensor(mat, dtype=torch.float32)
    mat -= torch.mean(mat)
    print("*" * 30)
    print(f'[Using Classes] {np.unique(np.array(label))}')
    print("*" * 30)
    # 计算余弦相似度矩阵
    mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
    A = mat_normalized @ mat_normalized.T
    A.to(device)

    for LAMBDA in LAMBDA_LIST:
        print(f'[Using LAMBDA] {LAMBDA}')
        reload(algo1)
        H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
        H.to(device)
        H_out, P_out, losses, losses_pure, times, H_list = algo1.matrixFac_col(X=A, itr=10000000, H=H, lmbda=LAMBDA, tol=1e-15)
        torch.norm(H_out - P_out)
        pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
        acc = ari.ari(label, pred_labels)
        nmi_score = nmi.nmi(label, pred_labels)
        print(f'[Accuracy] {acc}')
        print(f'[Mutual Information] {nmi_score}')

        # 更新平均值
        if LAMBDA in avg_acc:
            avg_acc[LAMBDA].append(acc)
        else:
            avg_acc[LAMBDA] = [acc]

        if LAMBDA in avg_nmi:
            avg_nmi[LAMBDA].append(nmi_score)
        else:
            avg_nmi[LAMBDA] = [nmi_score]

# 计算平均值
for LAMBDA in LAMBDA_LIST:
    avg_acc_value = np.mean(avg_acc[LAMBDA])
    avg_nmi_value = np.mean(avg_nmi[LAMBDA])
    print(f'Average Accuracy for LAMBDA={LAMBDA}: {avg_acc_value}')
    print(f'Average Mutual Information for LAMBDA={LAMBDA}: {avg_nmi_value}')